In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "smoke"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage2b"
SEED = 41
MODEL_NAME = ["dino_wm_pusht", "jepa_wm_pusht"]
ENVIRONMENT = "PushT"
HORIZONS = [1, 3, 6]
NUM_STATES = 24
ACTIONS_PER_STATE = 10

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage2b"
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
FEATURE_POOL_GRID = 4
BOOTSTRAP_REPS = 500
CV_REPEATS = 5
CANDIDATE_LIBRARY_SIZE = 22
PAIR_EFFECT_SCALE_MIN = 1e-6

if RUN_MODE == "full":
    NUM_STATES = 250
    ACTIONS_PER_STATE = 10
    BOOTSTRAP_REPS = 2000
    CV_REPEATS = 20
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == ["dino_wm_pusht", "jepa_wm_pusht"]
assert ENVIRONMENT == "PushT"
assert HORIZONS == [1, 3, 6]
assert 8 <= ACTIONS_PER_STATE <= 12


# Stage 2B: confirmatory counterfactual-faithfulness pilot

Stage 2 completed technically but its executable physical-regret endpoint
was floor-heavy: the fixed no-op action was the physical oracle in 71.7% of
real-model rows. The pre-specified incremental-validity interval crossed
zero, so the result was `INCONCLUSIVE` and did not authorize Stage 3.

This confirmatory revision changes only the intervention design:

- the block begins away from the task goal;
- the agent begins behind the block relative to the goal;
- a deterministic 22-sequence library is generated in state-relative
  coordinates;
- a fixed subset of 10 candidates is chosen before evaluation without
  consulting future simulator outcomes;
- the same two public checkpoints, metrics, grouped cross-validation, and
  state-clustered decision rule are retained.

The full run evaluates 250 exact simulator states, 10 fixed alternatives,
horizons 1/3/6, and both DINO-WM and JEPA-WM Push-T checkpoints. It is still
a simulator-only study and makes no claim about real-robot reliability.

Run all cells from a fresh GPU runtime. A 16 GB T4 is sufficient; an L4 or
A100 is recommended. Google Drive is optional. Intermediate shards are
resumable, and `stage2b_result_bundle.zip` downloads automatically after
success or a captured failure.


In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-matched torch and torchvision builds.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected.")


In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
INTERMEDIATE = OUT / "intermediate"
TRUTH_DIR = INTERMEDIATE / "truth"
MODEL_ROOT = INTERMEDIATE / "models"
LOG_DIR = OUT / "logs"
PLOT_DIR = OUT / "plots"
for path in [OUT, INTERMEDIATE, TRUTH_DIR, MODEL_ROOT, LOG_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage2")
log.info("Seeds set to %d", SEED)

def gpu_report(label):
    payload = {
        "label": label,
        "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    }
    log.info("GPU memory %s", payload)
    return payload

CONFIG = {
    "RUN_MODE": RUN_MODE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SEED": SEED,
    "MODEL_NAME": MODEL_NAME,
    "ENVIRONMENT": ENVIRONMENT,
    "HORIZONS": HORIZONS,
    "NUM_STATES": NUM_STATES,
    "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "REPO_URL": REPO_URL,
    "REPO_COMMIT": REPO_COMMIT,
    "FRAMESKIP": FRAMESKIP,
    "FEATURE_POOL_GRID": FEATURE_POOL_GRID,
    "BOOTSTRAP_REPS": BOOTSTRAP_REPS,
    "CV_REPEATS": CV_REPEATS,
    "CANDIDATE_LIBRARY_SIZE": CANDIDATE_LIBRARY_SIZE,
    "PAIR_EFFECT_SCALE_MIN": PAIR_EFFECT_SCALE_MIN,
    "pinned_dependencies": PINNED,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()
CONFIG["run_signature"] = RUN_SIGNATURE
config_path = OUT / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    if previous.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "OUTPUT_DIR contains a different configuration; choose a new OUTPUT_DIR."
        )
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
(OUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

PIPELINE_FAILED = False
FAILURE_MESSAGE = ""

def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)

gpu_report("startup")


In [ ]:
# Core simulator, metric, model-loading, and statistical helpers.
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value

def write_json(path, payload):
    Path(path).write_text(json.dumps(json_ready(payload), indent=2) + "\n")

def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        raise ValueError(f"no rows for {path}")
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def pool_visual(visual, grid=FEATURE_POOL_GRID):
    value = visual.detach().float().cpu().numpy()[..., 0, :, :, :]
    height, width, dim = value.shape[-3:]
    if height % grid or width % grid:
        raise ValueError(f"feature grid {(height, width)} is not divisible by {grid}")
    fh, fw = height // grid, width // grid
    value = value.reshape(*value.shape[:-3], grid, fh, grid, fw, dim)
    value = value.mean(axis=(-4, -2))
    return value.reshape(*value.shape[:-3], -1)

def pair_indices(n_actions):
    left, right = [], []
    for i in range(n_actions):
        for j in range(i + 1, n_actions):
            left.append(i)
            right.append(j)
    return np.asarray(left), np.asarray(right)

def counterfactual_arrays(truth, prediction, eps=1e-12):
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    if truth.shape != prediction.shape or truth.ndim != 3:
        raise ValueError(f"expected matching [action,horizon,feature], got {truth.shape}")
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(0, 2)))
    common = np.mean(errors, axis=0)
    common_rmse = np.sqrt(np.mean(common**2, axis=-1))
    centered = errors - common[None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(0, 2)))
    left, right = pair_indices(truth.shape[0])
    dy = truth[left] - truth[right]
    dy_hat = prediction[left] - prediction[right]
    pair_error = dy_hat - dy
    pair_rmse = np.sqrt(np.mean(pair_error**2, axis=-1))
    pair_scale = np.sqrt(np.mean(dy**2, axis=-1))
    pair_normalized = pair_rmse / np.maximum(pair_scale, eps)
    dot = np.sum(dy_hat * dy, axis=-1)
    denom = np.linalg.norm(dy_hat, axis=-1) * np.linalg.norm(dy, axis=-1)
    # A zero predicted intervention receives zero directional alignment.
    pair_cosine = np.divide(
        dot, denom, out=np.zeros_like(dot), where=denom > eps
    )
    aggregate_pair_rmse = np.sqrt(np.mean(pair_error**2, axis=(0, 2)))
    aggregate_scale = np.sqrt(np.mean(dy**2, axis=(0, 2)))
    aggregate_normalized = aggregate_pair_rmse / np.maximum(aggregate_scale, eps)
    aggregate_cosine = np.mean(pair_cosine, axis=0)
    expected_pair_mse = (
        2 * truth.shape[0] / (truth.shape[0] - 1)
    ) * action_dependent**2
    identity_residual = aggregate_pair_rmse**2 - expected_pair_mse
    return {
        "ordinary_rmse": ordinary,
        "common_mode_rmse": common_rmse,
        "action_dependent_rmse": action_dependent,
        "paired_effect_rmse": aggregate_pair_rmse,
        "ground_truth_effect_rms": aggregate_scale,
        "normalized_paired_effect_rmse": aggregate_normalized,
        "paired_effect_cosine": aggregate_cosine,
        "identity_residual": identity_residual,
        "pair_left": left,
        "pair_right": right,
        "pair_effect_rmse": pair_rmse,
        "pair_effect_scale": pair_scale,
        "pair_normalized_effect_rmse": pair_normalized,
        "pair_effect_cosine": pair_cosine,
    }

def ranking_arrays(true_cost, predicted_cost, eps=1e-12, tie=1e-9):
    true_cost = np.asarray(true_cost, dtype=np.float64)
    predicted_cost = np.asarray(predicted_cost, dtype=np.float64)
    if true_cost.shape != predicted_cost.shape or true_cost.ndim != 2:
        raise ValueError("costs must match with shape [action,horizon]")
    selected = np.argmin(predicted_cost, axis=0)
    oracle = np.argmin(true_cost, axis=0)
    horizon_index = np.arange(true_cost.shape[1])
    chosen = true_cost[selected, horizon_index]
    best = np.min(true_cost, axis=0)
    regret = chosen - best
    spread = np.max(true_cost, axis=0) - best
    normalized_regret = np.divide(
        regret,
        np.maximum(spread, eps),
        out=np.zeros_like(regret),
        where=spread > eps,
    )
    top1 = (chosen <= best + tie).astype(np.float64)
    left, right = pair_indices(true_cost.shape[0])
    true_delta = true_cost[left] - true_cost[right]
    pred_delta = predicted_cost[left] - predicted_cost[right]
    valid = np.abs(true_delta) > tie
    pair_credit = np.full_like(true_delta, np.nan)
    pair_credit[valid & (np.sign(true_delta) == np.sign(pred_delta))] = 1.0
    pair_credit[valid & (np.abs(pred_delta) <= tie)] = 0.5
    pair_credit[valid & np.isnan(pair_credit)] = 0.0
    pairwise_accuracy = np.nanmean(pair_credit, axis=0)
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": top1,
        "regret": regret,
        "normalized_regret": normalized_regret,
        "pairwise_accuracy": pairwise_accuracy,
        "pairwise_credit": pair_credit,
    }

def task_cost(states, goal=np.array([256.0, 256.0, np.pi / 4])):
    states = np.asarray(states)
    angle = np.arctan2(
        np.sin(states[..., 4] - goal[2]),
        np.cos(states[..., 4] - goal[2]),
    )
    pieces = np.concatenate(
        [(states[..., 2:4] - goal[:2]) / 512.0, (angle / np.pi)[..., None]],
        axis=-1,
    )
    return np.linalg.norm(pieces, axis=-1)

def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError("zero direction")
    return vector / norm

def rotate_vector(vector, degrees):
    radians = np.deg2rad(degrees)
    matrix = np.array(
        [
            [np.cos(radians), -np.sin(radians)],
            [np.sin(radians), np.cos(radians)],
        ]
    )
    return matrix @ vector

def build_states(count):
    rng = np.random.default_rng(SEED)
    states, strata = [], []
    goal_xy = np.array([256.0, 256.0])
    for index in range(count):
        radial_distance = rng.uniform(90.0, 130.0)
        polar_angle = rng.uniform(-np.pi, np.pi)
        block = goal_xy + radial_distance * np.array(
            [np.cos(polar_angle), np.sin(polar_angle)]
        )
        push_direction = unit_vector(goal_xy - block)
        if index % 2 == 0:
            agent_distance = rng.uniform(60.0, 68.0)
            stratum = "near"
        else:
            agent_distance = rng.uniform(72.0, 80.0)
            stratum = "far"
        agent = block - agent_distance * push_direction
        if np.any(agent < 35.0) or np.any(agent > 477.0):
            raise AssertionError(f"agent out of bounds: {agent}")
        block_angle = rng.uniform(-0.60, 0.60)
        states.append(
            [agent[0], agent[1], block[0], block[1], block_angle, 0.0, 0.0]
        )
        strata.append(stratum)
    return np.asarray(states, dtype=np.float64), np.asarray(strata)

def candidate_library(state, primitive_steps):
    goal_xy = np.array([256.0, 256.0])
    push_direction = unit_vector(goal_xy - np.asarray(state)[2:4])
    specifications = [("noop", 0.0, 0)]
    specifications.extend(
        (f"direct_{duration}", 0.0, duration)
        for duration in [8, 12, 16, 20, 24, 30]
    )
    for angle in [-20.0, 20.0]:
        for duration in [12, 18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-40.0, 40.0]:
        for duration in [18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-70.0, 70.0, 140.0, -140.0, 180.0]:
        specifications.append((f"angle_{angle:+.0f}_24", angle, 24))
    if len(specifications) != CANDIDATE_LIBRARY_SIZE:
        raise AssertionError("candidate-library size changed")

    sequences = []
    for _, angle, duration in specifications:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        if duration:
            sequence[:duration] = (
                0.14 * rotate_vector(push_direction, angle)
            ).astype(np.float32)
        sequences.append(sequence)
    return np.stack(sequences), [item[0] for item in specifications]

def fixed_candidate_indices():
    # Frozen before confirmatory model evaluation. These indices cover
    # no-op, multiple push durations, and symmetric angular deviations.
    chosen = np.asarray([0, 2, 4, 6, 8, 11, 14, 16, 17, 18], dtype=np.int64)
    if len(chosen) != ACTIONS_PER_STATE or len(np.unique(chosen)) != len(chosen):
        raise AssertionError("invalid fixed candidate subset")
    if np.min(chosen) < 0 or np.max(chosen) >= CANDIDATE_LIBRARY_SIZE:
        raise AssertionError("fixed candidate index outside library")
    return chosen

def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT], check=True)
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in Push-T predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    configs = [
        repo / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in configs:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo

def make_environment(repo):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv
    return PushTEnv(
        with_velocity=True,
        with_target=True,
        render_size=224,
        relative=True,
        action_scale=100,
    )

def reset_env(env, state, seed):
    env.seed(seed)
    env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
    observation, restored = env.reset()
    return {
        "visual": np.asarray(observation["visual"]).copy(),
        "proprio": np.asarray(observation["proprio"]).copy(),
    }, np.asarray(restored).copy()

def rollout_branch(env, state, primitive_actions, horizons, seed):
    observation0, restored = reset_env(env, state, seed)
    wanted = set(horizons)
    observations, states, contacts, coverages = {}, {}, {}, {}
    cumulative_contacts = 0
    for step, action in enumerate(primitive_actions, start=1):
        observation, _, _, info = env.step(action)
        cumulative_contacts += int(info.get("n_contacts", 0))
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = {
                    "visual": np.asarray(observation["visual"]).copy(),
                    "proprio": np.asarray(observation["proprio"]).copy(),
                }
                states[horizon] = np.asarray(info["state"]).copy()
                contacts[horizon] = cumulative_contacts
                coverages[horizon] = float(info["final_coverage"])
    if wanted != set(observations):
        raise RuntimeError(f"missing simulator horizons: {wanted - set(observations)}")
    return observation0, restored, observations, states, contacts, coverages

def exact_restore_test(env, state, actions, repeats=3):
    endpoints, images, diagnostics = [], [], []
    for _ in range(repeats):
        initial, _, _, states, contacts, coverages = rollout_branch(
            env, state, actions, [max(HORIZONS)], SEED + 5000
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        diagnostics.append((contacts[max(HORIZONS)], coverages[max(HORIZONS)]))
    result = {
        "repeats": repeats,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            diagnostics[0] == item for item in diagnostics[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(np.max(np.abs(endpoints[0] - item)) for item in endpoints[1:])
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result

def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim == 5:
        pass
    else:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}

def state_metrics(truth_features, prediction_features, physical_cost, coverage_cost, contacts):
    variants = {
        "real": prediction_features,
        "action_blind": np.broadcast_to(
            np.mean(prediction_features, axis=0, keepdims=True),
            prediction_features.shape,
        ).copy(),
        "action_shuffled": prediction_features[
            np.roll(np.arange(prediction_features.shape[0]), 1)
        ],
    }
    output = {"variant_names": np.asarray(list(variants))}
    latent_true_cost = np.sqrt(
        np.mean((truth_features - CURRENT_GOAL_FEATURE[None, None, :]) ** 2, axis=-1)
    )
    output["latent_true_cost"] = latent_true_cost
    output["physical_true_cost"] = physical_cost
    output["coverage_true_cost"] = coverage_cost
    output["contacts"] = contacts
    metric_names = None
    stored = {}
    for variant_name, predicted in variants.items():
        metrics = counterfactual_arrays(truth_features, predicted)
        if metric_names is None:
            metric_names = list(metrics)
        predicted_latent_cost = np.sqrt(
            np.mean(
                (predicted - CURRENT_GOAL_FEATURE[None, None, :]) ** 2,
                axis=-1,
            )
        )
        rank_latent = ranking_arrays(latent_true_cost, predicted_latent_cost)
        rank_physical = ranking_arrays(physical_cost, predicted_latent_cost)
        rank_coverage = ranking_arrays(coverage_cost, predicted_latent_cost)
        stored[variant_name] = {
            **metrics,
            "latent_predicted_cost": predicted_latent_cost,
            **{f"latent_{key}": value for key, value in rank_latent.items()},
            **{f"physical_{key}": value for key, value in rank_physical.items()},
            **{f"coverage_{key}": value for key, value in rank_coverage.items()},
        }
    for key in stored["real"]:
        output[key] = np.stack([stored[name][key] for name in variants])
    return output


In [ ]:
# Phase A — execute a fixed, state-relative candidate set in the simulator.
def generate_simulator_truth():
    repo = configure_repo()
    env = make_environment(repo)
    states, state_strata = build_states(NUM_STATES)
    primitive_steps = max(HORIZONS) * FRAMESKIP
    goal_state = np.array(
        [80.0, 450.0, 256.0, 256.0, np.pi / 4, 0.0, 0.0]
    )
    goal_observation, _ = reset_env(env, goal_state, SEED + 7000)

    first_library, library_labels = candidate_library(states[0], primitive_steps)
    fixed_indices = fixed_candidate_indices()
    restore = exact_restore_test(
        env, states[0], first_library[fixed_indices[1]], repeats=3
    )
    write_json(OUT / "restore_test.json", restore)
    log.info("Exact restoration: %s", restore)

    for state_index, state in enumerate(states):
        state_path = TRUTH_DIR / f"state_{state_index:04d}.npz"
        if state_path.exists():
            log.info("Simulator resume: keeping %s", state_path.name)
            continue
        library, labels = candidate_library(state, primitive_steps)
        if labels != library_labels:
            raise AssertionError("candidate labels vary by state")
        selected = fixed_candidate_indices()
        visuals, proprios, endpoints = [], [], []
        contacts, coverages, initials = [], [], []
        for actions in library[selected]:
            initial, restored, observations, states_by_h, contacts_by_h, coverage_by_h = (
                rollout_branch(
                    env,
                    state,
                    actions,
                    HORIZONS,
                    SEED + state_index,
                )
            )
            initials.append(initial["visual"])
            visuals.append([observations[h]["visual"] for h in HORIZONS])
            proprios.append([observations[h]["proprio"] for h in HORIZONS])
            endpoints.append([states_by_h[h] for h in HORIZONS])
            contacts.append([contacts_by_h[h] for h in HORIZONS])
            coverages.append([coverage_by_h[h] for h in HORIZONS])
        if not all(np.array_equal(initials[0], item) for item in initials[1:]):
            raise AssertionError(
                f"branch initial render mismatch at state {state_index}"
            )

        endpoint_array = np.asarray(endpoints)
        contact_array = np.asarray(contacts, dtype=np.int32)
        physical_cost = task_cost(endpoint_array)
        atomic_npz(
            state_path,
            initial_state=state,
            design_stratum=state_strata[state_index],
            initial_visual=initials[0],
            initial_proprio=reset_env(
                env, state, SEED + state_index
            )[0]["proprio"],
            selected_actions=library[selected],
            selected_library_indices=selected,
            future_visual=np.asarray(visuals, dtype=np.uint8),
            future_proprio=np.asarray(proprios, dtype=np.float32),
            endpoint_states=endpoint_array.astype(np.float32),
            contacts=contact_array,
            coverage=np.asarray(coverages, dtype=np.float32),
            physical_cost=physical_cost.astype(np.float32),
        )
        write_json(
            OUT / "simulator_progress.json",
            {
                "run_signature": RUN_SIGNATURE,
                "completed_states": state_index + 1,
                "total_states": NUM_STATES,
                "last_file": state_path.name,
            },
        )
        log.info("Simulator state %d/%d", state_index + 1, NUM_STATES)

    action_bank, selected_indices = [], []
    all_costs, all_contacts = [], []
    for state_index in range(NUM_STATES):
        with np.load(TRUTH_DIR / f"state_{state_index:04d}.npz") as shard:
            action_bank.append(shard["selected_actions"])
            selected_indices.append(shard["selected_library_indices"])
            all_costs.append(shard["physical_cost"])
            all_contacts.append(shard["contacts"])
    action_bank = np.asarray(action_bank, dtype=np.float32)
    selected_indices = np.asarray(selected_indices, dtype=np.int64)
    all_costs = np.asarray(all_costs, dtype=np.float64)
    all_contacts = np.asarray(all_contacts, dtype=np.int32)

    oracle = np.argmin(all_costs, axis=1)
    spread = np.max(all_costs, axis=1) - np.min(all_costs, axis=1)
    no_op_regret = all_costs[:, 0] - np.min(all_costs, axis=1)
    left, right = pair_indices(ACTIONS_PER_STATE)
    pair_contact_count = (
        (all_contacts[:, left, :] > 0).astype(int)
        + (all_contacts[:, right, :] > 0).astype(int)
    )
    design_summary = {
        "candidate_library_size": CANDIDATE_LIBRARY_SIZE,
        "selected_actions_per_state": ACTIONS_PER_STATE,
        "selection_protocol": (
            "fixed state-relative subset frozen before model evaluation; "
            "future simulator outcomes are not used for candidate selection"
        ),
        "no_op_oracle_fraction_by_horizon": np.mean(
            oracle == 0, axis=0
        ).tolist(),
        "no_op_positive_regret_fraction_by_horizon": np.mean(
            no_op_regret > 1e-9, axis=0
        ).tolist(),
        "median_physical_cost_spread_by_horizon": np.median(
            spread, axis=0
        ).tolist(),
        "minimum_physical_cost_spread_by_horizon": np.min(
            spread, axis=0
        ).tolist(),
        "contact_fraction_by_horizon": np.mean(
            all_contacts > 0, axis=(0, 1)
        ).tolist(),
        "pair_contact_counts": {
            label: int(np.sum(pair_contact_count == index))
            for index, label in enumerate(["neither", "one", "both"])
        },
    }
    if design_summary["no_op_oracle_fraction_by_horizon"][-1] >= 0.20:
        raise AssertionError("final-horizon no-op oracle remains degenerate")
    if design_summary["no_op_positive_regret_fraction_by_horizon"][-1] <= 0.80:
        raise AssertionError("final-horizon no-op regret is insufficient")
    if design_summary["median_physical_cost_spread_by_horizon"][-1] <= 0.08:
        raise AssertionError("final-horizon physical cost spread is insufficient")
    if not all(
        design_summary["pair_contact_counts"][label] > 0
        for label in ["neither", "one", "both"]
    ):
        raise AssertionError("candidate design is missing a contact stratum")
    write_json(OUT / "candidate_design_summary.json", design_summary)

    atomic_npz(
        OUT / "design.npz",
        states=states,
        state_strata=state_strata,
        action_bank=action_bank,
        selected_library_indices=selected_indices,
        candidate_library_labels=np.asarray(library_labels),
        goal_state=goal_state,
        goal_visual=goal_observation["visual"],
        goal_proprio=goal_observation["proprio"],
    )

    import pymunk
    write_json(
        OUT / "environment.json",
        {
            "name": "JEPA-WMs bundled PushTEnv",
            "repository": REPO_URL,
            "commit": REPO_COMMIT,
            "pymunk": pymunk.version,
            "relative_actions": True,
            "action_scale": 100,
            "with_velocity": True,
            "frameskip": FRAMESKIP,
            "branch_protocol": "fresh simulator space per action branch",
            "candidate_protocol": "fixed state-relative action subset",
            "action_pair_contact_strata": ["neither", "one", "both"],
        },
    )
    return repo

if not PIPELINE_FAILED:
    try:
        REPO = generate_simulator_truth()
    except Exception:
        record_failure("simulator_truth")


In [ ]:
# Phase B — evaluate both checkpoints sequentially and save derived state shards.
def evaluate_models():
    global CURRENT_GOAL_FEATURE
    repo = configure_repo()
    with np.load(OUT / "design.npz") as design:
        action_bank = design["action_bank"]
        goal_visual = design["goal_visual"]
        goal_proprio = design["goal_proprio"]
    max_horizon = max(HORIZONS)

    for model_index, model_name in enumerate(MODEL_NAME):
        model_dir = MODEL_ROOT / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        torch.cuda.reset_peak_memory_stats()
        gpu_report(f"{model_name}_before_load")
        model, preprocessor = torch.hub.load(
            str(repo),
            model_name,
            source="local",
            pretrained=True,
            device="cuda:0",
            trust_repo=True,
        )
        model.eval()
        gpu_report(f"{model_name}_after_load")
        with torch.inference_mode():
            goal_encoded = model.encode(
                to_model_observation(goal_visual, goal_proprio)
            )
            CURRENT_GOAL_FEATURE = pool_visual(goal_encoded["visual"])[0, 0]

        horizon_index = torch.tensor(HORIZONS, dtype=torch.long, device="cuda")

        for state_index in range(NUM_STATES):
            output_path = model_dir / f"state_{state_index:04d}.npz"
            if output_path.exists():
                log.info("%s resume: keeping %s", model_name, output_path.name)
                continue
            with np.load(TRUTH_DIR / f"state_{state_index:04d}.npz") as truth_shard:
                initial_visual = truth_shard["initial_visual"]
                initial_proprio = truth_shard["initial_proprio"]
                future_visual = truth_shard["future_visual"]
                future_proprio = truth_shard["future_proprio"]
                physical_cost = truth_shard["physical_cost"].astype(np.float64)
                coverage_cost = 1.0 - truth_shard["coverage"].astype(np.float64)
                contacts = truth_shard["contacts"]
            chunks = torch.from_numpy(
                action_bank[state_index].reshape(
                    ACTIONS_PER_STATE, max_horizon, FRAMESKIP, 2
                )
            ).float()
            normalized_chunks = preprocessor.normalize_actions(chunks)
            model_actions = (
                normalized_chunks.reshape(ACTIONS_PER_STATE, max_horizon, -1)
                .permute(1, 0, 2)
                .contiguous()
                .cuda()
            )
            with torch.inference_mode():
                initial_encoded = model.encode(
                    to_model_observation(initial_visual, initial_proprio)
                )
                truth_encoded = model.encode(
                    to_model_observation(future_visual, future_proprio)
                )
                truth_features = pool_visual(truth_encoded["visual"])
                predicted_encoded = model.unroll(initial_encoded, model_actions)
                predicted_selected = predicted_encoded["visual"].index_select(
                    0, horizon_index
                )
                predicted_features = np.moveaxis(
                    pool_visual(predicted_selected), 0, 1
                )
            metrics = state_metrics(
                truth_features,
                predicted_features,
                physical_cost,
                coverage_cost,
                contacts,
            )
            atomic_npz(output_path, **metrics)
            write_json(
                OUT / f"{model_name}_progress.json",
                {
                    "run_signature": RUN_SIGNATURE,
                    "model": model_name,
                    "completed_states": state_index + 1,
                    "total_states": NUM_STATES,
                    "last_file": output_path.name,
                },
            )
            log.info("%s state %d/%d", model_name, state_index + 1, NUM_STATES)
            if (state_index + 1) % 25 == 0 or RUN_MODE == "smoke":
                gpu_report(f"{model_name}_state_{state_index:04d}")

        del model, preprocessor, goal_encoded, initial_encoded
        gc.collect()
        torch.cuda.empty_cache()
        gpu_report(f"{model_name}_released")

if not PIPELINE_FAILED:
    try:
        evaluate_models()
    except Exception:
        record_failure("model_evaluation")


In [ ]:
# Phase C — aggregate, run state-grouped analyses, plot, package, and download.
AGGREGATE_KEYS = [
    "ordinary_rmse",
    "common_mode_rmse",
    "action_dependent_rmse",
    "paired_effect_rmse",
    "ground_truth_effect_rms",
    "normalized_paired_effect_rmse",
    "paired_effect_cosine",
    "identity_residual",
]
OBJECTIVES = ["latent", "physical", "coverage"]

def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if len(unique) == 0:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": int(repetitions),
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.nanmean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.nanmean(values)),
        "low": float(np.nanquantile(draws, 0.025)),
        "high": float(np.nanquantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }

def ridge_predict(x_train, y_train, x_test, ridge=1e-3):
    mean = np.mean(x_train, axis=0)
    scale = np.std(x_train, axis=0)
    scale[scale < 1e-12] = 1.0
    train = (x_train - mean) / scale
    test = (x_test - mean) / scale
    train = np.column_stack([np.ones(len(train)), train])
    test = np.column_stack([np.ones(len(test)), test])
    penalty = np.eye(train.shape[1]) * ridge
    penalty[0, 0] = 0.0
    coef = np.linalg.solve(
        train.T @ train + penalty,
        train.T @ y_train,
    )
    return test @ coef

def grouped_oof(y, x, groups, seed, folds=5):
    unique = np.unique(groups).copy()
    rng = np.random.default_rng(seed)
    rng.shuffle(unique)
    fold_groups = np.array_split(unique, folds)
    prediction = np.empty_like(y, dtype=np.float64)
    for held_out in fold_groups:
        test = np.isin(groups, held_out)
        prediction[test] = ridge_predict(x[~test], y[~test], x[test])
    return prediction

def score_predictions(y, prediction):
    sse = np.sum((y - prediction) ** 2)
    total = np.sum((y - np.mean(y)) ** 2)
    return {
        "r2": float(1.0 - sse / total),
        "rmse": float(np.sqrt(np.mean((y - prediction) ** 2))),
    }

def bootstrap_oof_improvement(y, base, full, groups, repetitions, seed):
    unique = np.unique(groups)
    index_by_group = [np.flatnonzero(groups == group) for group in unique]
    rng = np.random.default_rng(seed)
    draws_mse, draws_r2 = [], []
    for _ in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        rows = np.concatenate([index_by_group[item] for item in sampled])
        y_draw = y[rows]
        base_mse = np.mean((y_draw - base[rows]) ** 2)
        full_mse = np.mean((y_draw - full[rows]) ** 2)
        variance = np.mean((y_draw - np.mean(y_draw)) ** 2)
        draws_mse.append(base_mse - full_mse)
        draws_r2.append((base_mse - full_mse) / max(variance, 1e-12))
    return {
        "mse_improvement": {
            "estimate": float(
                np.mean((y - base) ** 2) - np.mean((y - full) ** 2)
            ),
            "low": float(np.quantile(draws_mse, 0.025)),
            "high": float(np.quantile(draws_mse, 0.975)),
        },
        "delta_r2": {
            "estimate": float(
                score_predictions(y, full)["r2"]
                - score_predictions(y, base)["r2"]
            ),
            "low": float(np.quantile(draws_r2, 0.025)),
            "high": float(np.quantile(draws_r2, 0.975)),
        },
    }

def aggregate_outputs():
    with np.load(OUT / "design.npz") as design:
        state_strata = design["state_strata"].astype(str)
        action_bank = design["action_bank"]
    unit_rows, pair_rows = [], []
    left, right = pair_indices(ACTIONS_PER_STATE)
    variant_names = None

    for model_name in MODEL_NAME:
        for state_index in range(NUM_STATES):
            with np.load(
                MODEL_ROOT / model_name / f"state_{state_index:04d}.npz"
            ) as shard:
                if variant_names is None:
                    variant_names = shard["variant_names"].astype(str).tolist()
                contacts = shard["contacts"]
                contact_flags = contacts > 0
                for variant_index, variant in enumerate(variant_names):
                    for horizon_index, horizon in enumerate(HORIZONS):
                        row = {
                            "state_id": state_index,
                            "model": model_name,
                            "variant": variant,
                            "horizon": horizon,
                            "design_stratum": state_strata[state_index],
                            "contact_fraction": float(
                                np.mean(contact_flags[:, horizon_index])
                            ),
                        }
                        for key in AGGREGATE_KEYS:
                            row[key] = float(
                                shard[key][variant_index, horizon_index]
                            )
                        for objective in OBJECTIVES:
                            for key in [
                                "selected_action",
                                "oracle_action",
                                "top1_correct",
                                "regret",
                                "normalized_regret",
                                "pairwise_accuracy",
                            ]:
                                value = shard[f"{objective}_{key}"][
                                    variant_index, horizon_index
                                ]
                                if key in {"selected_action", "oracle_action"}:
                                    value = int(value)
                                else:
                                    value = float(value)
                                row[f"{objective}_{key}"] = value
                        unit_rows.append(row)

                    for pair_index, (pair_left, pair_right) in enumerate(
                        zip(left, right)
                    ):
                        for horizon_index, horizon in enumerate(HORIZONS):
                            contact_count = int(
                                contact_flags[pair_left, horizon_index]
                            ) + int(contact_flags[pair_right, horizon_index])
                            contact_stratum = ["neither", "one", "both"][
                                contact_count
                            ]
                            action_prefix = horizon * FRAMESKIP
                            action_distance = float(
                                np.sqrt(
                                    np.mean(
                                        (
                                            action_bank[
                                                state_index, pair_left, :action_prefix
                                            ]
                                            - action_bank[
                                                state_index, pair_right, :action_prefix
                                            ]
                                        )
                                        ** 2
                                    )
                                )
                            )
                            pair_row = {
                                "state_id": state_index,
                                "model": model_name,
                                "variant": variant,
                                "horizon": horizon,
                                "design_stratum": state_strata[state_index],
                                "pair_left": int(pair_left),
                                "pair_right": int(pair_right),
                                "contact_stratum": contact_stratum,
                                "action_distance": action_distance,
                                "effect_rmse": float(
                                    shard["pair_effect_rmse"][
                                        variant_index,
                                        pair_index,
                                        horizon_index,
                                    ]
                                ),
                                "effect_scale": float(
                                    shard["pair_effect_scale"][
                                        variant_index,
                                        pair_index,
                                        horizon_index,
                                    ]
                                ),
                                "normalized_effect_rmse": float(
                                    shard["pair_normalized_effect_rmse"][
                                        variant_index,
                                        pair_index,
                                        horizon_index,
                                    ]
                                ),
                                "effect_cosine": float(
                                    shard["pair_effect_cosine"][
                                        variant_index,
                                        pair_index,
                                        horizon_index,
                                    ]
                                ),
                            }
                            for objective in OBJECTIVES:
                                pair_row[f"{objective}_ranking_credit"] = float(
                                    shard[f"{objective}_pairwise_credit"][
                                        variant_index,
                                        pair_index,
                                        horizon_index,
                                    ]
                                )
                            pair_rows.append(pair_row)
    write_csv(OUT / "unit_metrics.csv", unit_rows)
    write_csv(OUT / "pair_metrics.csv", pair_rows)
    return unit_rows, pair_rows, variant_names

def run_incremental_analysis(unit_rows):
    real = [row for row in unit_rows if row["variant"] == "real"]
    groups = np.asarray([row["state_id"] for row in real])
    ordinary = np.asarray([row["ordinary_rmse"] for row in real])
    magnitude = np.asarray(
        [row["normalized_paired_effect_rmse"] for row in real]
    )
    direction = np.asarray(
        [1.0 - row["paired_effect_cosine"] for row in real]
    )
    if not np.all(np.isfinite(direction)):
        raise AssertionError("real-model direction metric contains non-finite values")
    horizons = np.asarray([row["horizon"] for row in real])
    models = np.asarray([MODEL_NAME.index(row["model"]) for row in real])
    effect_scale = np.asarray(
        [row["ground_truth_effect_rms"] for row in real]
    )
    contact_fraction = np.asarray(
        [row["contact_fraction"] for row in real]
    )
    far = np.asarray([row["design_stratum"] == "far" for row in real], dtype=float)
    horizon_dummy = np.column_stack(
        [(horizons == value).astype(float) for value in HORIZONS[1:]]
    )
    nuisance = np.column_stack(
        [horizon_dummy, models, effect_scale, contact_fraction, far]
    )
    base_x = np.column_stack([ordinary, nuisance])
    blocks = {
        "magnitude": magnitude[:, None],
        "direction": direction[:, None],
        "joint": np.column_stack([magnitude, direction]),
    }
    outcomes = {
        "physical_normalized_regret": np.asarray(
            [row["physical_normalized_regret"] for row in real]
        ),
        "latent_normalized_regret": np.asarray(
            [row["latent_normalized_regret"] for row in real]
        ),
        "coverage_normalized_regret": np.asarray(
            [row["coverage_normalized_regret"] for row in real]
        ),
        "physical_pairwise_error": 1.0
        - np.asarray([row["physical_pairwise_accuracy"] for row in real]),
    }
    cv_rows, bootstrap = [], {}
    for outcome_name, y_all in outcomes.items():
        valid = np.isfinite(y_all)
        y = y_all[valid]
        outcome_base_x = base_x[valid]
        outcome_groups = groups[valid]
        if len(np.unique(outcome_groups)) < 10:
            raise AssertionError(
                f"insufficient finite state clusters for {outcome_name}"
            )
        bootstrap[outcome_name] = {}
        for block_name, block in blocks.items():
            full_x = np.column_stack([outcome_base_x, block[valid]])
            for seed in range(CV_REPEATS):
                base_pred = grouped_oof(
                    y, outcome_base_x, outcome_groups, seed
                )
                full_pred = grouped_oof(y, full_x, outcome_groups, seed)
                base_score = score_predictions(y, base_pred)
                full_score = score_predictions(y, full_pred)
                cv_rows.append(
                    {
                        "outcome": outcome_name,
                        "counterfactual_block": block_name,
                        "seed": seed,
                        "base_r2": base_score["r2"],
                        "full_r2": full_score["r2"],
                        "delta_r2": full_score["r2"] - base_score["r2"],
                        "base_rmse": base_score["rmse"],
                        "full_rmse": full_score["rmse"],
                        "rmse_improvement": base_score["rmse"]
                        - full_score["rmse"],
                    }
                )
            base_fixed = grouped_oof(
                y, outcome_base_x, outcome_groups, 0
            )
            full_fixed = grouped_oof(y, full_x, outcome_groups, 0)
            bootstrap[outcome_name][block_name] = bootstrap_oof_improvement(
                y,
                base_fixed,
                full_fixed,
                outcome_groups,
                BOOTSTRAP_REPS,
                SEED + len(cv_rows),
            )
    write_csv(OUT / "incremental_validity.csv", cv_rows)
    write_json(OUT / "incremental_validity_bootstrap.json", bootstrap)
    return cv_rows, bootstrap

def summarize(unit_rows, pair_rows, cv_rows, bootstrap):
    summary_rows = []
    for model_name in MODEL_NAME:
        for variant in ["real", "action_blind", "action_shuffled"]:
            for horizon in HORIZONS:
                selected = [
                    row
                    for row in unit_rows
                    if row["model"] == model_name
                    and row["variant"] == variant
                    and row["horizon"] == horizon
                ]
                groups = np.asarray([row["state_id"] for row in selected])
                result = {
                    "model": model_name,
                    "variant": variant,
                    "horizon": horizon,
                    "num_states": len(selected),
                }
                for key in [
                    "ordinary_rmse",
                    "normalized_paired_effect_rmse",
                    "paired_effect_cosine",
                    "physical_top1_correct",
                    "physical_normalized_regret",
                    "latent_top1_correct",
                    "latent_normalized_regret",
                ]:
                    values = np.asarray([row[key] for row in selected])
                    interval = bootstrap_mean(
                        values,
                        groups,
                        BOOTSTRAP_REPS,
                        SEED + horizon + len(summary_rows),
                    )
                    result[key] = interval["estimate"]
                    result[f"{key}_low"] = interval["low"]
                    result[f"{key}_high"] = interval["high"]
                summary_rows.append(result)
    write_csv(OUT / "metrics_summary.csv", summary_rows)

    contact_rows = []
    for model_name in MODEL_NAME:
        for horizon in HORIZONS:
            for contact_stratum in ["neither", "one", "both"]:
                selected = [
                    row
                    for row in pair_rows
                    if row["model"] == model_name
                    and row["variant"] == "real"
                    and row["horizon"] == horizon
                    and row["contact_stratum"] == contact_stratum
                ]
                if not selected:
                    continue
                groups = np.asarray([row["state_id"] for row in selected])
                row = {
                    "model": model_name,
                    "horizon": horizon,
                    "contact_stratum": contact_stratum,
                    "num_pairs": len(selected),
                    "num_states": len(np.unique(groups)),
                }
                for key in [
                    "effect_rmse",
                    "effect_scale",
                    "normalized_effect_rmse",
                    "effect_cosine",
                    "physical_ranking_credit",
                    "latent_ranking_credit",
                ]:
                    values = np.asarray([item[key] for item in selected])
                    finite = np.isfinite(values)
                    if key in {"normalized_effect_rmse", "effect_cosine"}:
                        finite &= (
                            np.asarray(
                                [item["effect_scale"] for item in selected]
                            )
                            >= PAIR_EFFECT_SCALE_MIN
                        )
                    row[f"{key}_num_pairs"] = int(np.sum(finite))
                    row[f"{key}_num_states"] = int(
                        len(np.unique(groups[finite]))
                    )
                    interval = bootstrap_mean(
                        values[finite],
                        groups[finite],
                        BOOTSTRAP_REPS,
                        SEED + horizon + len(contact_rows),
                    )
                    row[key] = interval["estimate"]
                    row[f"{key}_low"] = interval["low"]
                    row[f"{key}_high"] = interval["high"]
                contact_rows.append(row)
    write_csv(OUT / "contact_strata_summary.csv", contact_rows)

    primary_bootstrap = bootstrap["physical_normalized_regret"]["joint"]
    primary_cv = [
        row
        for row in cv_rows
        if row["outcome"] == "physical_normalized_regret"
        and row["counterfactual_block"] == "joint"
    ]
    median_delta = float(np.median([row["delta_r2"] for row in primary_cv]))
    positive_fraction = float(
        np.mean([row["delta_r2"] > 0 for row in primary_cv])
    )
    low = primary_bootstrap["mse_improvement"]["low"]
    high = primary_bootstrap["mse_improvement"]["high"]
    if low > 0 and median_delta > 0:
        decision = "NONREDUNDANT_SIGNAL"
    elif high < 0 and median_delta < 0:
        decision = "NEGATIVE_SIGNAL"
    else:
        decision = "INCONCLUSIVE"
    decision_payload = {
        "status": decision,
        "primary_outcome": "physical_normalized_regret",
        "counterfactual_block": [
            "normalized_paired_effect_rmse",
            "one_minus_paired_effect_cosine",
        ],
        "base_predictors": [
            "ordinary_rmse",
            "horizon",
            "model",
            "ground_truth_effect_rms",
            "contact_fraction",
            "design_stratum",
        ],
        "median_repeated_cv_delta_r2": median_delta,
        "fraction_repeats_positive_delta_r2": positive_fraction,
        "cluster_bootstrap": primary_bootstrap,
        "interpretation_guardrail": (
            "This is simulator evidence about controlled interventions and "
            "executable Push-T planning, not real-robot reliability."
        ),
    }
    write_json(OUT / "stage2_decision.json", decision_payload)
    write_json(
        OUT / "metrics_summary.json",
        {
            "status": "SUCCESS",
            "run_mode": RUN_MODE,
            "num_states": NUM_STATES,
            "models": MODEL_NAME,
            "horizons": HORIZONS,
            "variants": ["real", "action_blind", "action_shuffled"],
            "decision": decision_payload,
            "summary_rows": summary_rows,
            "contact_strata_rows": contact_rows,
        },
    )
    return summary_rows, contact_rows, decision_payload

def make_plots(unit_rows, contact_rows, cv_rows):
    real = [row for row in unit_rows if row["variant"] == "real"]
    fig, axes = plt.subplots(1, len(MODEL_NAME), figsize=(12, 4), squeeze=False)
    for model_index, model_name in enumerate(MODEL_NAME):
        selected = [row for row in real if row["model"] == model_name]
        scatter = axes[0, model_index].scatter(
            [row["ordinary_rmse"] for row in selected],
            [row["normalized_paired_effect_rmse"] for row in selected],
            c=[row["physical_normalized_regret"] for row in selected],
            cmap="viridis",
            s=18,
            alpha=0.75,
        )
        axes[0, model_index].set_title(model_name)
        axes[0, model_index].set_xlabel("ordinary latent RMSE")
        axes[0, model_index].set_ylabel("normalized paired error")
    fig.colorbar(scatter, ax=axes.ravel().tolist(), label="physical normalized regret")
    fig.subplots_adjust(wspace=0.3, right=0.86)
    fig.savefig(PLOT_DIR / "ordinary_vs_counterfactual.png", dpi=180)
    plt.close(fig)

    labels = ["neither", "one", "both"]
    fig, axes = plt.subplots(1, len(MODEL_NAME), figsize=(12, 4), squeeze=False)
    for model_index, model_name in enumerate(MODEL_NAME):
        selected = [
            row
            for row in contact_rows
            if row["model"] == model_name and row["horizon"] == max(HORIZONS)
        ]
        lookup = {row["contact_stratum"]: row for row in selected}
        x = np.arange(len(labels))
        means = [
            lookup[label]["effect_rmse"] if label in lookup else np.nan
            for label in labels
        ]
        axes[0, model_index].bar(x, means)
        axes[0, model_index].set_xticks(x, labels)
        axes[0, model_index].set_title(f"{model_name}, horizon {max(HORIZONS)}")
        axes[0, model_index].set_ylabel("pair effect RMSE")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "contact_strata.png", dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4))
    blocks = ["magnitude", "direction", "joint"]
    values = [
        [
            row["delta_r2"]
            for row in cv_rows
            if row["outcome"] == "physical_normalized_regret"
            and row["counterfactual_block"] == block
        ]
        for block in blocks
    ]
    ax.boxplot(values, tick_labels=blocks)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_ylabel("held-out delta R²")
    ax.set_title("Incremental prediction of physical normalized regret")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "incremental_validity.png", dpi=180)
    plt.close(fig)

def checkpoint_manifest():
    candidates = []
    for root in [Path(os.environ["HF_HOME"]), Path(os.environ["TORCH_HOME"])]:
        if root.exists():
            for path in root.rglob("*"):
                if path.is_file() and (
                    "dino_wm_pusht" in path.name
                    or "jepa_wm_pusht" in path.name
                    or "dinov2_vits14" in path.name
                    or "dinov2_vits14" in str(path)
                ):
                    candidates.append(path)
    manifest = [
        {
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(set(candidates))
    ]
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "models": MODEL_NAME,
            "repository": "facebook/jepa-wms",
            "encoder": "facebookresearch/dinov2:dinov2_vits14",
            "dataset_downloaded": False,
            "cached_files": manifest,
        },
    )

def intermediate_manifest():
    files = [path for path in INTERMEDIATE.rglob("*") if path.is_file()]
    write_json(
        OUT / "intermediate_manifest.json",
        {
            "excluded_from_result_zip": True,
            "file_count": len(files),
            "total_bytes": sum(path.stat().st_size for path in files),
            "truth_shards": len(list(TRUTH_DIR.glob("state_*.npz"))),
            "model_shards": {
                model: len(list((MODEL_ROOT / model).glob("state_*.npz")))
                for model in MODEL_NAME
            },
        },
    )

def execute_analysis():
    unit_rows, pair_rows, variants = aggregate_outputs()
    cv_rows, bootstrap = run_incremental_analysis(unit_rows)
    summary_rows, contact_rows, decision = summarize(
        unit_rows, pair_rows, cv_rows, bootstrap
    )
    make_plots(unit_rows, contact_rows, cv_rows)
    checkpoint_manifest()
    intermediate_manifest()
    max_identity = max(abs(row["identity_residual"]) for row in unit_rows)
    real_contact_strata = {
        row["contact_stratum"]
        for row in pair_rows
        if row["variant"] == "real"
    }
    if max_identity > 1e-8:
        raise AssertionError(f"paired metric identity residual too large: {max_identity}")
    if not {"neither", "one"}.issubset(real_contact_strata):
        raise AssertionError(
            f"insufficient contact strata observed: {sorted(real_contact_strata)}"
        )
    (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")
    gpu_report("analysis_complete")
    return decision

def package_results():
    result_zip = OUT.parent / "stage2b_result_bundle.zip"
    included = []
    with zipfile.ZipFile(
        result_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as archive:
        for path in sorted(OUT.rglob("*")):
            if not path.is_file():
                continue
            relative = path.relative_to(OUT)
            if relative.parts and relative.parts[0] == "intermediate":
                continue
            archive.write(path, arcname=str(relative))
            included.append(str(relative))
    write_json(
        OUT / "result_zip_manifest.json",
        {
            "archive": str(result_zip),
            "included_before_manifest_write": included,
            "intermediate_excluded": True,
        },
    )
    # Re-open once to include the manifest created immediately above.
    with zipfile.ZipFile(
        result_zip, "a", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as archive:
        archive.write(
            OUT / "result_zip_manifest.json",
            arcname="result_zip_manifest.json",
        )
    print(f"RESULT ZIP: {result_zip}")
    return result_zip

RUN_STATUS = "FAILED" if PIPELINE_FAILED else "PENDING"
if not PIPELINE_FAILED:
    try:
        STAGE2_DECISION = execute_analysis()
        RUN_STATUS = "SUCCESS"
    except Exception:
        record_failure("analysis")
        RUN_STATUS = "FAILED"

RESULT_ZIP = package_results()
print("RUN_STATUS:", RUN_STATUS)
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
    print(f"Automatic download requested: {RESULT_ZIP.name}")
except Exception as download_exc:
    print("Automatic download unavailable; use the Colab Files pane.", download_exc)


In [ ]:
# Final compact status check. The preceding cell already requested the download.
if RUN_STATUS == "SUCCESS":
    required = [
        "config.json",
        "versions.json",
        "environment.json",
        "restore_test.json",
        "candidate_design_summary.json",
        "checkpoints_manifest.json",
        "unit_metrics.csv",
        "pair_metrics.csv",
        "metrics_summary.csv",
        "metrics_summary.json",
        "contact_strata_summary.csv",
        "incremental_validity.csv",
        "incremental_validity_bootstrap.json",
        "stage2_decision.json",
        "FAILURE_TRACE.txt",
        "logs/run.log",
        "plots/ordinary_vs_counterfactual.png",
        "plots/contact_strata.png",
        "plots/incremental_validity.png",
    ]
    missing = [name for name in required if not (OUT / name).exists()]
    if missing:
        raise AssertionError(f"result bundle is missing: {missing}")
    assert (OUT / "FAILURE_TRACE.txt").read_text().strip() == "NONE"
    decision = json.loads((OUT / "stage2_decision.json").read_text())
    print(json.dumps(decision, indent=2))
    print(f"Sanity checks passed. Return: {RESULT_ZIP.name}")
else:
    print("Stage 2B captured a failure.")
    print(f"Return {RESULT_ZIP.name}; it contains FAILURE_TRACE.txt and logs.")
